In [1]:
import os
import json
from docx import Document
from typing import List, Dict, Optional
import dotenv
from google import genai
from google.api_core import retry
import pandas as pd
import numpy as np
from datetime import datetime

In [38]:
ApplicationExample = Dict[str, str]

def load_json(path: str) -> List[ApplicationExample]:
    """
    Load JSON file, return data as list of dicts.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    
def get_doc_text(path):
    """
    Extract text from document at path.
    """
    return [['','[B]'][any([j in p.paragraph_format.element.xml
                            for j in ['<w:numPr>', 'w:val="ListBullet"']])]
            + p.text
            for p in Document(path).paragraphs]

def get_line_index(findtxt, intxt):
    """
    Return line number within findtxt that begins with intxt.
    """
    return (z[0] if (z:=[i for i,j in enumerate(intxt) 
                         if j.upper().startswith(findtxt.upper())]) 
            else -1)
    
def normalize_text(text: str) -> str:
    """
    Normalize text by replacing special characters and trimming whitespace.
    """
    if text is None:
        return ""
    repl_map = [("\u201c", '"'), 
                ("\u201d", '"'), 
                ("\u2018", "'"), 
                ("\u2019", "'"), 
                ("\u2013", "-"), 
                ("\u2014", "-")]
    return [(z:=(z if i else text).replace(j,k)) 
            for i,(j,k) in enumerate(repl_map)][-1].strip()

def past_example_components(past_examples: List[Dict]) -> List[Dict]:
    """
    Get details of past applications from JSON file, read 
    """
    doc_components = []
    for e in past_examples:
        resume_text = get_doc_text(e['resume_path'])
        cover_text = get_doc_text(e['cover_path'])
        doc_components += [{
            'r_title': resume_text[2:3],
            'r_hl_skills': resume_text[3:4],
            'r_intro': resume_text[4:get_line_index('Technical Skills', resume_text)-1],
            'r_tech': resume_text[get_line_index('Technical Skills', resume_text)+1:
                                  get_line_index('Areas of Expertise', resume_text)-1],
            'r_expert': resume_text[get_line_index('Areas of Expertise', resume_text)+1:
                                    get_line_index('Professional Experience', resume_text)-1],
            'r_effo': resume_text[get_line_index('Data Science Director', resume_text)+2:
                                  get_line_index('dunnhumbyUSA', resume_text)-1],
            'r_dusa_vp': resume_text[get_line_index('Vice President', resume_text)+2:
                                     get_line_index('Analysis Director', resume_text)-1],
            'r_dusa_dir': resume_text[get_line_index('Analysis Director', resume_text)+2:
                                      get_line_index('Senior Analyst', resume_text)-1],
            'r_dusa_sa': resume_text[get_line_index('Senior Analyst', resume_text)+2:
                                     get_line_index('dunnhumby ', resume_text)-1],
            'r_duk': resume_text[get_line_index('Marketing Analyst', resume_text)+2:
                                 get_line_index('Education', resume_text)-1],
            'c_p1': cover_text[(s:=get_line_index('Dear', cover_text)+1): (s:=s+1)], 
            'c_p2': cover_text[s: (s:=s+1)], 
            'c_p2b': cover_text[s: (s:=s+sum([i[:3]=='[B]' for i in cover_text[s:]]))], 
            'c_p3': cover_text[s: (s:=s+1)], 
            'c_p4': cover_text[s: (s:=s+1)], 
        }]
    return [{k: [normalize_text(l) for l in e[k]] 
             for k in e} 
            for e in doc_components]
    
def get_job_details() -> Dict[str, str]:
    """
    Gather job posting details from user inputs, clean and return as dict.
    """
    company_input = title_input = loc_input = details_input = None
    while not title_input:
        title_input = input('Enter job title')
    while not company_input:
        company_input = input('Enter name of hiring company')
    while not loc_input:
        loc_input = input('Enter location of hiring company')
    while not details_input:
        details_input = input('Enter job description')
        
    return {
        "role": normalize_text(title_input),
        "company": normalize_text(company_input),
        "location": normalize_text(loc_input),
        "apply_date": datetime.today().strftime("%m%d%Y"),
        "job_posting": normalize_text(details_input),
        }

def create_genai_client():
    """
    Set up access to Google Gen AI models.
    """
    if not "GOOGLE_API_KEY" in os.environ:
        dotenv.load_dotenv()
    is_retriable = lambda e: (
        isinstance(e, genai.errors.APIError) and e.code in {429, 503}
    )
    genai.models.Models.generate_content = retry.Retry(
        predicate=is_retriable
    )(genai.models.Models.generate_content)

    return genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def create_gemini_chat(client,
                       modelname: str = "gemini-3.5-flash") -> str:
    """
    Generate text using the Gemini API.
    """
    config = genai.types.GenerateContentConfig(
        temperature=0.6,
        top_p=0.95
    )
    return client.chats.create(model=modelname,
                               config=config,
                               history=[])
    
def proper(txt):
    """
    Apply sentence case to string, but preserve acronyms as all-caps.
    """
    f,t = 0,1
    while f<len(txt):
        while t<len(txt) and (65<=ord(txt[t])<=90 or 97<=ord(txt[t])<=122):
            t+=1
        if txt[f:t]!=txt[f:t].upper():
            txt = (txt[:f]
                   +(txt[f:t].lower() if f else txt[f:t].capitalize())
                   +txt[t:])
        t = (f:=t+1)+1
    return txt

In [ ]:
# Load details from past applications from json file
past_examples = load_json('job_application_examples.json')

# Remove weird quotes and dashes, and leading/trailing spaces
doc_components = past_example_components(past_examples)

# Get job posting details
job_details = get_job_details()

In [8]:
# Initialize Gemini chat
client = create_genai_client()
chat = create_gemini_chat(client)

In [ ]:
def initialize_chat(client, prompt, job_details):
    """
    Start a new Gemini chat, provide job details and basic instructions
    """
    chat = create_gemini_chat(client)
    chat.send_message(prompt
                      .replace('<ROLE>', job_details['role'])
                      .replace('<COMPANY>', job_details['company'])
                      .replace('<LOCATION>', job_details['location'])
                      .replace('<DETAILS>', job_details['job_posting'])
                      )

In [ ]:

    
    prompt = f"""
I want you to help me customize my resume and write a cover letter for a job application.  
Please remember the details from the following job posting.  Throughout this chat I will refer to this as the "new" job posting:
Job title: {job_details['role']}
Name of hiring company: {job_details['company']}
Location of hiring company: {job_details['location']}
Job description:
{job_details['job_posting']}
(END OF JOB DESCRIPTION TEXT)
Here are some standing instructions I would like you to follow when generating text throughout this chat:
- When writing new text, do not invent experiences or skills not mentioned in the previous examples.
- The returned text should have one space after each comma and two spaces after each period.
- Do not use em dashes.
(END OF STANDING INSTRUCTIONS)
You don't need to give a response to this prompt.
""".strip()

    
    

In [9]:
# Set the scene and share job posting
prompt = f"""
I want you to help me customize my resume and write a cover letter for a job application.  
Please remember the details from the following job posting.  Throughout this chat I will refer to this as the "new" job posting:
Job title: {job_details['role']}
Name of hiring company: {job_details['company']}
Location of hiring company: {job_details['location']}
Job description:
{job_details['job_posting']}
(END OF JOB DESCRIPTION TEXT)
Here are some standing instructions I would like you to follow when generating text throughout this chat:
- When writing new text, do not invent experiences or skills not mentioned in the previous examples.
- The returned text should have one space after each comma and two spaces after each period.
- Do not use em dashes.
(END OF STANDING INSTRUCTIONS)
You don't need to give a response to this prompt.
""".strip()

_ = chat.send_message(prompt)

# Set up empty list to gather outputs
llm_outputs = {}

In [11]:
# Generate a resume title
source_examples = '\n'.join({i.upper() 
                             for i in sum([e['r_title'] 
                                           for e in doc_components],[])})
prompt = f"""
Under the "PREVIOUS EXAMPLES" heading below, I've provided a list of resume titles that I have used in previous job 
applications.  In each case, the title text was selected to align with the details of the position being applied for.  
For example, for a role requiring a combination of business analytics, machine learning and team leadership, a fitting 
resume title might be "DATA SCIENCE & ANALYTICS LEADER".

Your task is to generate the title text that I should use when applying for the new job posting.  To minimize the amount 
of new text I have to review, please use text from the PREVIOUS EXAMPLES list wherever possible.  Concretely, I would
like you to use the following process:
1. From the PREVIOUS EXAMPLES list, identify the example that most closely aligns with the job posting.
2. If the selected example seems appropriate for the job posting, return the string "UNCHANGED", followed by a 
newline, followed by the selected text with no changes.  Then do nothing else.
3. If the selected example does not seem appropriate for the job posting, but you think that appropriate text can
be composed by combining parts taken from different examples, return the string "REMIXED", followed by a 
newline, followed by the text you composed.  Then do nothing else.
4. If you do not see a way to compose appropriate text using the method in step 3, try to generate text by combining
elements from the PREVIOUS EXAMPLES list with newly generated text, still using the former as much as possible.  Return 
the string "NEWTEXT", followed by a newline, followed by the text you composed.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['r_title'] = chat.send_message(prompt).text
llm_outputs['r_title']

'REMIXED\nSENIOR INSIGHTS & ANALYTICS LEADER'

In [ ]:
# Generate headline skills
source_examples = '\n'.join({s.strip().upper()
                             for e in doc_components
                             for s in e['r_hl_skills'][0].split('|')})
prompt = f"""
Under the "PREVIOUS EXAMPLES" heading below, I've provided a list of "headline" skills lists - which appear immediately
underneath the title of my resume - that I have used in previous job applications.  In each case, 3 items were selected
to align with the details of the position being applied for, and combined into a single string with pipe separators.  For 
example, for a role involving business analytics and machine learning that requires some skill in translating business 
problems into analytical methodology, the "headline" skills might be: "Problem framing | Decision Intelligence | Machine Learning".

Your task is to generate the "headline" skills list that I should use when applying for the new job posting.  As with 
the previous task, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, select 3 items that most closely align with the job posting.  Avoid selecting 
items that are too similar in meaning or relate to similar aspects of the role (e.g. "ACTIONABLE INTELLIGENCE"
and "DECISION INTELLIGENCE").
2. If the selected items adequately represent the full range of skills needed for the role, return the string 
"UNCHANGED", followed by a newline, followed by the pipe-delimited string.  Then do nothing else.
3. If some important aspect of the role is not well represented by any set of 3 items from the PREVIOUS EXAMPLES list,
try to generate a suitable "headline" skills list by combining elements from the PREVIOUS EXAMPLES list with newly 
generated text, still using the former as much as possible.  Return the string "NEWTEXT", followed by a newline, 
followed by the pipe-delimited string.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['r_hl_skills'] = chat.send_message(prompt).text
llm_outputs['r_hl_skills']

'UNCHANGED\nRETAIL STRATEGY | STRATEGIC DECISION SUPPORT | ACTIONABLE INSIGHT'

In [13]:
# Generate intro paragraph
source_examples = '\n\n'.join([e['r_intro'][0] 
                               for e in doc_components])
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of examples of the introductory paragraph of my resume used in previous job 
applications.  In each case, the text was customized to align with the details of the position being applied for.
For example, where customer analytics was a major focus of the role, the paragraph emphasizes this.

Your task is to generate an introductory paragraph for the resume I should use when applying for the new job posting.
As with previous tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, identify the paragraph that most closely aligns with the job posting.
2. If the selected paragraph seems appropriate for the job posting, return the string "UNCHANGED", followed by a 
newline, followed by the full text of the selected paragraph with NO changes.  Then do nothing else.
3. If the selected paragraph does not seem appropriate for the job posting, but you think you can generate a suitable 
paragraph by combining parts of different examples (i.e. full sentences and clauses, not individual words or small groups 
of words), return the string "REMIXED", followed by a newline, followed by the paragraph you composed.  Then do nothing else.
4. If a suitable paragraph cannot be generated using the method in step 3, try to generate it by combining elements 
from the PREVIOUS EXAMPLES list with newly generated text, still using the former as much as possible.  Return the string 
"NEWTEXT", followed by a newline, followed by the paragraph you composed.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['r_intro'] = chat.send_message(prompt).text
llm_outputs['r_intro']

'UNCHANGED\nAnalytics professional with over 25 years of experience helping retailers and consumer packaged goods companies understand customer behavior, track and analyze key performance metrics, and realize opportunities for growth.  Extensive experience leveraging loyalty, transactional, demographic, and behavioral data to inform pricing, promotion, assortment, personalization, and category growth initiatives.  Trusted partner to executive stakeholders, with a track record of translating complex data into compelling narratives and actionable recommendations.  Combines deep grocery retail expertise with hands-on analytical capability and a passion for mentoring and developing future insights leaders.'

In [ ]:
# Generate tech skills
exnum, sk, py = zip(*[[n]+[i.replace(')','').strip() 
                           for i in (s.split('(')+[''])[:2]]
                      for n, e in enumerate(doc_components)
                      for s in '|'.join(e['r_tech']).split('|')])

# (Get unique list of skills, preserve average rank ordering from source docs)
skord = [(z:=1 if i==0 or j!=exnum[i-1] else z+1)
         for i,j in enumerate(exnum)]
source_examples = '\n'.join(pd.DataFrame({'skill': sk, 'skrank': skord})
                                .groupby('skill').mean()
                                .sort_values(by='skrank').index.tolist())

# (Separate list of Python packages, similarly ranked)
source_examples_py = '\n'.join(pd.DataFrame([[k.strip(),j]
                                             for i in py if i
                                             for j,k in enumerate(i.split(','))],
                                            columns=['pyskill', 'skrank'])
                                   .groupby('pyskill').mean()
                                   .sort_values(by='skrank').index.tolist())
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of technical tools and programming languages that have been listed as technical
skills in my resume for previous job applications.  In each case, the ordering and the level of detail were adjusted
to align with the details of the position being applied for.  For example, for a job with a greater focus on leadership 
than hands-on work, I would tend to exclude items that have a relatively narrow range of use cases, unless they are
specifically mentioned in the job posting.  The ordering of the list below reflects the order in which these items 
have been listed in previous resumes - for example, SQL and Python are almost always listed first.

Under "PYTHON PACKAGES" below is a list of Python packages that were included as a "sub-list" within these technical 
skills lists.  Again, these are selected and ordered to align to the job details.

These lists are combined into a single string as follows:
1. Items in the main list are joined with pipe delimiters.
2. The Python packages list is inserted after "Python", as a comma-separated list in parentheses.
3. When the length of the resulting string exceeds 120 characters, some of the pipe delimiters are replaced with newlines
to create a multi-line string in which each line is less than 120 characters long.

For example:
SQL | Python (NumPy, Pandas, PySpark, Matplotlib, Scikit-learn, TensorFlow, PyTorch) | Databricks | Microsoft Azure
Power BI | SAS | R | Git/GitHub | Excel/VBA | Microsoft Office

Your task is to generate the technical skills list I should use when applying for the new job posting.  As with previous 
tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, select an appropriate list of items to align with the job posting as described above.
Avoid selecting items that are too similar in meaning to other included items or parts of other items (e.g. "Excel" and
"Excel/VBA").  Keep them in the same order as listed below, unless a different order would better fit the job posting.
2. From the PYTHON PACKAGES list, select an appropriate list of items to align with the job posting as described above.
Keep them in the same order as listed below, unless a different order would better fit the job posting.
3. Create the output string as described above.
4. If you think that the job description demands knowledge of an important technical tool or programming language that
is not included in the output string, return the string "MISSING:", followed by the missing skill(s), followed by a 
newline, followed by the output string.
5. Otherwise, return the string "GOODLIST", followed by a newline, followed by the output string.

PREVIOUS EXAMPLES:
{source_examples}

PYTHON PACKAGES:
{source_examples_py}
""".strip()
llm_outputs['r_tech'] = chat.send_message(prompt).text
llm_outputs['r_tech']

'MISSING: Tableau, Stratum, Market 6, Circana, Nielsen\nSQL | Python (NumPy, Pandas, Matplotlib) | Power BI | Excel | Microsoft Office'

In [16]:
# Generate areas of expertise
source_examples = '\n'.join({proper(s.strip())
                             for e in doc_components
                             for s in '|'.join(e['r_expert']).split('|')})
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of items that have been listed as "areas of expertise" in my resume for previous 
job applications.  In each case, between 22 and 28 items were selected to align with the details of the position being 
applied for.  These are combined into a single string with pipe separators.  When the length of this string exceeds 120 
characters, some of the pipe delimiters are replaced with newlines to create a multi-line string in which each line is 
less than 120 characters long.

For example:
Problem framing | Exploratory analysis | Hypothesis development | Experimental design | A/B testing | KPI development
Machine learning | Clustering & segmentation | Predictive modeling | Data visualization | Dashboard development
Data discovery | Data transformation | Strategic decision support | Analytical storytelling | Stakeholder partnership
Cross-functional collaboration | Coaching and mentoring | Team leadership | Interviewing | Analytics governance

Your task is to generate the areas of expertise list I should use when applying for the new job posting.  As with previous 
tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, select between 22 and 28 items that collectively align with the job posting.  Avoid 
selecting items that have similar meanings or that whose meanings overlap significantly (e.g. "Data storytelling" and
"Analytical storytelling").  Try to put the items in a somewhat logical order, keeping items related to project 
leadership, techniques, team working and other subsets together.
2. Create the output string as described above.
3. If you think that the job description demands area(s) of expertise that are not included in the output string, return 
the string "MISSING:", followed by the missing area(s), followed by a newline, followed by the output string.
4. Otherwise, return the string "GOODLIST", followed by a newline, followed by the output string.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['r_expert'] = chat.send_message(prompt).text
llm_outputs['r_expert']

'MISSING: Forecasting\nProblem framing | Business requirements translation | Strategic insights | Strategic decision support\nBusiness intelligence | Large-scale retail data | Customer behavior analysis | Market research\nPromotion effectiveness | Assortment optimization | Exploratory analysis | Trend analysis | Ad hoc analysis\nData discovery | Data transformation | KPI metrics & measurement | Dashboards & reporting | Analytical storytelling\nData quality and governance | Project planning and management | Stakeholder partnership\nCross-functional collaboration | Team leadership | Coaching and mentoring | Process improvement'

In [17]:
# Generate role achievements bullets
for job in ['effo', 'dusa_vp', 'dusa_dir', 'dusa_sa', 'duk']:
    print(job)
    source_examples = '\n-----\n'.join([l 
                                        for l in open(f'bullets_{job}.txt', 'r')
                                                     .read().strip().split('\n\n')])
    prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of items that have been listed as my achievements in a specific past role in my
resume, used for previous job applications.  For each achievement, the version of the description used was adjusted to 
align to the job description.  The order in which the achievements were listed was also customized, with those most 
relevant to the job being applied for being placed closest to the top of the list.  In the list below, descriptions that 
relate to the same achievement are grouped together, with the string '-----' used to separate different groups.

Your task is to generate the list of achievements I should use when applying for the new job posting.  As with previous 
tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. Considering the achievements first, determine in which order they should appear in the output.  Place those most relevant 
to the job posting closest to the top of the list.
2. After having determined the order in which the achievements should be listed, select the description that should
be used to represent each one.  Use the following sub-process for each achievement:
(a) Select the version of the achievement description from the PREVIOUS EXAMPLES list that most closely aligns with the 
job posting.
(b) If the version you selected seems appropriate for the job posting, return the string "UNCHANGED", followed by a 
newline, followed by the text of the selected description with NO changes.
(c) If the version you selected does not seem appropriate for the job posting, generate new text that better aligns to the
job posting without materially changing its meaning, in a style consistent with previous examples.  Return the string 
"REWRITTEN", followed by a newline, followed by the rewritten text.
When finished, you should have generated one description for each group of descriptions in the PREVIOUS EXAMPLES list.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
    llm_outputs[f'r_{job}'] = chat.send_message(prompt).text
    print(llm_outputs[f'r_{job}'])

effo
UNCHANGED
Led strategic shopper analytics initiatives supporting Kroger's personalized marketing and loyalty programs, using customer behavioral data to improve engagement, strengthen loyalty, and drive category growth.

UNCHANGED
Coached and developed teams of data scientists as part of 84.51°'s formal performance management process, including setting goals, conducting regular check-ins and evaluating performance.

UNCHANGED
Developed automated reporting and analytics processes to evaluate targeted promotional performance, partnering with stakeholders to define standardized KPIs, metrics, and visualizations while reducing delivery timelines by 70%.

UNCHANGED
Developed methodology for automated definition of product coupons, using vector embeddings to identify product groupings and sales metrics to define offer details, improving offer relevance by up to 20% while reducing manual effort.

UNCHANGED
Designed and deployed a self-service analytics application that automated recurrin

In [ ]:
# Generate cover letter P1 (purpose, high-level experience)
source_examples = '\n\n'.join([e['c_p1'][0] 
                               for e in doc_components])
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of examples of the first paragraph of the cover letter used for several
previous job applications.  In each case, the text is customized to align with the details of the position being applied 
for.  The paragraph usually consists of an opening sentence stating that I'm applying for the open role (quoting the job
title), followed by a high-level overview of my experience and how it is relevant to the position being applied for.

Your task is to generate the first paragraph of the cover letter I should use when applying for the new job posting.  
As with previous tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, identify the paragraph that most closely aligns with the job posting.
2. If the selected paragraph (with job title and name of the hiring company updated) seems appropriate for the job posting,
return the string "UNCHANGED", followed by a newline, followed by the full text of the selected paragraph with NO changes
(other than to the job title and name of the hiring company).  Then do nothing else.
3. If the selected paragraph does not seem appropriate for the job posting, but you think you can generate a suitable 
paragraph by combining parts of different examples (i.e. full sentences and clauses, not individual words or small groups 
of words), return the string "REMIXED", followed by a newline, followed by the paragraph you composed.  Then do nothing else.
4. If a suitable paragraph cannot be generated using the method in step 3, try to generate it by combining elements 
from the PREVIOUS EXAMPLES list with newly generated text, still using the former as much as possible.  Return the string 
"NEWTEXT", followed by a newline, followed by the paragraph you composed.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['c_p1'] = chat.send_message(prompt).text
llm_outputs['c_p1']

'UNCHANGED\nI am writing to apply for the Senior Sales Insights Manager, Kroger position at Harvest Group.  With more than 25 years of experience helping retailers and consumer packaged goods companies understand shopper behavior and translate insights into business growth, I bring a combination of deep grocery expertise, advanced analytical capability, and strategic partnership experience that aligns strongly with the requirements of this role.'

In [ ]:
# Generate cover letter P2 (experience from past roles)
source_examples = '\n\n'.join([e['c_p2'][0] 
                               for e in doc_components])
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of examples of the second paragraph of the cover letter used for several
previous job applications.  In each case, the text is customized to align with the details of the position being applied 
for.  The paragraph usually starts by describing the breadth of subject matter areas that my previous roles have touched
upon, emphasizing areas relevant to the new job posting.  Then it usually describes aspects that I particularly enjoy or 
that I'm adept in that made me successful in past roles and are also relevant to the job being applied for.  In cases 
where the job being applied for represents a major change from my past experience (e.g. a different industry or sector),
I often use this paragraph to acknowledge the change and emphasize the areas of commonality that help to justify my applying
for the new role.  The paragraph always ends with "Examples of work I have successfully delivered include:", as it is 
followed by a bulleted list of past achievements from past roles.

Your task is to generate the second paragraph of the cover letter I should use when applying for the new job posting.  
As with previous tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, identify the paragraph that most closely aligns with the job posting.
2. If the selected paragraph seems appropriate for the job posting, return the string "UNCHANGED", followed by a newline, 
followed by the full text of the selected paragraph with NO changes.  Then do nothing else.
3. If the selected paragraph does not seem appropriate for the job posting, but you think you can generate a suitable 
paragraph by combining parts of different examples (i.e. full sentences and clauses, not individual words or small groups 
of words), return the string "REMIXED", followed by a newline, followed by the paragraph you composed.  Then do nothing else.
4. If a suitable paragraph cannot be generated using the method in step 3, try to generate it by combining elements 
from the PREVIOUS EXAMPLES list with newly generated text, still using the former as much as possible.  Return the string 
"NEWTEXT", followed by a newline, followed by the paragraph you composed.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['c_p2'] = chat.send_message(prompt).text
llm_outputs['c_p2']

'UNCHANGED\nDuring my career at dunnhumby and 84.51°, I regularly served as the primary analytics partner for business teams, helping them define meaningful performance measures, challenge assumptions, and develop data-driven strategies.  While the subject matter varied broadly across areas such as pricing, promotions, customer engagement, and loyalty, the common thread for all analytical initiatives was working collaboratively with stakeholders to understand their objectives, identifying the right analytical approach, and translating complex data into practical recommendations.  Examples of work I have successfully delivered include:'

In [ ]:
# Generate cover letter P2 bullets (achievements in past roles)
source_examples = '\n'.join({b.replace('[B]','')
                             for e in doc_components
                             for b in e['c_p2b']})
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of items that have been listed as specific examples of work achievements in the 
cover letter used for several previous job applications.  In each case, the achievements were selected to align with the 
expectations of the role being applied for, showcasing the kind of positive impact I could be expected to deliver in the 
new role.  The wording is also adjusted to reflect the new posting, e.g. using key words and phrases from the job description
where possible and beneficial, without over- or understating the achievement as previously written.  (Note: As a result of
this re-wording, the same achievement may be represented multiple times in the PREVIOUS EXAMPLES list.)

Your task is to generate the examples of work achievements I should use when applying for the new job posting, that will
appear as a bulleted list under the second paragraph of the cover letter (directly following the phrase "Examples of work 
I have successfully delivered include:").  As with previous tasks, please use text from the PREVIOUS EXAMPLES list 
wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, select between 3 and 5 items that collectively align with the job posting.  Avoid 
selecting more than one item relating to the same achievement, and try to select examples that cover the full range of 
capabilities required by the new job posting.  Order these items by how well they align to the job description, with the 
most relevant ones first.
2. If you think any of the achievements mentioned in my resume but are not in the PREVIOUS EXAMPLES list would be
particularly impactful or relevant when applying for the new job posting, you can add it to the selected items or replace
one of the existing selected items with it.
3. For each achievement on the list:
(a) If the achievement was selected in step 2 above (i.e. not taken from the PREVIOUS EXAMPLES list), generate an impactful
description that aligns to the details of the new job posting and does not simply repeat the wording in my resume.  Return 
the string "NEWBULLET", followed by a newline, followed by the new text.
(b) If the achievement was selected from the PREVIOUS EXAMPLES and seems appropriate for the job posting, return the string 
"UNCHANGED", followed by a newline, followed by the text of the selected description with NO changes.
(c) If the achievement was selected from the PREVIOUS EXAMPLES and does not seem appropriate for the job posting, generate 
new text that better aligns to the job posting without materially changing its meaning, in a style consistent with previous
examples.  Return the string "REWRITTEN", followed by a newline, followed by the rewritten text.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['c_p2b'] = chat.send_message(prompt).text
llm_outputs['c_p2b']

'UNCHANGED\nCreating KPI reporting solutions that enabled business leaders to identify the underlying drivers of sales performance, including customer engagement, transaction behavior, pricing, and visit frequency.\n\nUNCHANGED\nDeveloping standardized methodology and metrics for the evaluation of targeted marketing campaigns at Kroger, improving consistency and transparency of results while reducing delivery timelines by 70%.\n\nUNCHANGED\nLeading the enhancement of personalized offer targeting models used for Kroger promotional campaigns, deploying updated models that drove a 10% improvement in customer response.\n\nUNCHANGED\nDeveloping analytical methods that incorporated customer shopping behavior into store layout decisions, using basket analysis to recommend product adjacencies that improved the shopping experience and supported business objectives.'

In [45]:
# Generate cover letter P3 (skills)
source_examples = '\n\n'.join([e['c_p3'][0] 
                               for e in doc_components])
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of examples of the third paragraph of the cover letter used for several
previous job applications.  In each case, the text is customized to align with the details of the position being applied 
for.  The paragraph usually starts by describing my technical skills, focusing mostly on skills that are relevant to the 
new job posting.  It may emphasize tools, programming languages and/or specific analytical techniques, depending on the 
details of the job being applied for.  Then it describes softer skills, again focusing on areas that are likely to be 
important for the job being applied for - either being mentioned specifically in the job posting, or implied by the 
job title or details of the position.

Your task is to generate the third paragraph of the cover letter I should use when applying for the new job posting.  
As with previous tasks, please use text from the PREVIOUS EXAMPLES list wherever possible.  Use the following process:
1. From the PREVIOUS EXAMPLES list, identify the paragraph that most closely aligns with the job posting.
2. If the selected paragraph seems appropriate for the job posting, return the string "UNCHANGED", followed by a newline, 
followed by the full text of the selected paragraph with NO changes.  Then do nothing else.
3. If the selected paragraph does not seem appropriate for the job posting, but you think you can generate a suitable 
paragraph by combining parts of different examples (i.e. full sentences and clauses, not individual words or small groups 
of words), return the string "REMIXED", followed by a newline, followed by the paragraph you composed.  Then do nothing else.
4. If a suitable paragraph cannot be generated using the method in step 3, try to generate it by combining elements 
from the PREVIOUS EXAMPLES list with newly generated text, still using the former as much as possible.  Return the string 
"NEWTEXT", followed by a newline, followed by the paragraph you composed.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['c_p3'] = chat.send_message(prompt).text
llm_outputs['c_p3']

'UNCHANGED\nAs a hands-on practitioner, I am highly proficient with tools used to query, transform, analyze, and visualize large datasets, including SQL, Python, SAS, Excel, and Power BI.  I enjoy working across functional teams with complementary skill sets, building trusted relationships with stakeholders, and helping business teams become more confident and effective users of data.  I also have extensive experience overseeing and mentoring analytics teams, fostering analytical rigor and helping others to strengthen their analytical capabilities and consulting skills.'

In [46]:
# Generate cover letter P4 (reason, signoff)
source_examples = '\n\n'.join([e['c_p4'][0] 
                               for e in doc_components])
print(source_examples)

I'm strongly drawn to Allstate's emphasis on truly understanding its customers' needs, and its commitment to driving informed decision-making using data science.  This aligns well with my background in customer-centric consulting, and I'm confident that I can make a meaningful impact as part of the Allstate team.  I greatly look forward to discussing the opportunity with you in greater depth.  Thank you for your time and consideration.

I am particularly excited by the opportunity to contribute to next-generation personalization and AI-driven customer experiences through modern recommendation systems, building on my experience and my desire to continue exploring and discovering value in the data science field.  I would welcome the opportunity to discuss how my skills and experience can contribute to the success of your team.  Thank you for your time and consideration.

I'm attracted to DICK'S Sporting Goods' emphasis on cross-functional team working and a product-oriented approach to a

In [47]:
prompt = f"""
Under "PREVIOUS EXAMPLES" below is a list of examples of the fourth paragraph of the cover letter used for several
previous job applications.  In each case, the text is customized to align with the details of the position being applied 
for.  The paragraph starts by describing a reason for my personal interest in the role and/or hiring company, usually in
terms of how specific aspects align well with my experience, values and interests.  Then it expresses confidence that I
would perform well and add value in the role, and eagerness to discuss the opportunity in greater depth.  It usually ends 
with "Thank you for your time and consideration.".

Your task is to generate the fourth paragraph of the cover letter I should use when applying for the new job posting.
For this task I would like you to match the style of writing in the PREVIOUS EXAMPLES list, and re-use text where it makes
sense.  However, since this paragraph is intended to add a more personal touch, you should use a little more freedom to 
generate text that is more tailored to the role and the hiring company.  For reference, the job posting will often describe
aspects of the hiring company's values and culture, while you can get some idea of my personal values and interests from
the PREVIOUS EXAMPLES list.

For this task, simply return the text you composed for this paragraph.

PREVIOUS EXAMPLES:
{source_examples}
""".strip()
llm_outputs['c_p4'] = chat.send_message(prompt).text
llm_outputs['c_p4']

"I am deeply drawn to Harvest Group's connected approach to commerce and its commitment to pursuing excellence with humility, which perfectly aligns with my belief in the power of collaborative, relationship-driven analytics.  Having spent much of my career delivering shopper insights for Kroger, I am excited for the opportunity to leverage my deep retail expertise and help consumer brands win in today's dynamic marketplace.  I am confident that my background in CPG and Kroger analytics will allow me to make a meaningful impact as part of your team, and I look forward to discussing this opportunity with you in greater depth.  Thank you for your time and consideration."

In [692]:
# for i in {b.replace('[B]','') for e in doc_components for b in e['r_duk']}:
#     print(i)